# Semana 2: Tratamiento de Datos Faltantes, Imputación y Detección de Outliers

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Convertir la serie con gaps en una serie temporal diaria completa y limpia, lista para EDA avanzado y modelado.

### Contenido:
1. Importar librerías
2. Cargar y preparar datos (replicar pipeline Week 1)
3. Reindexar a frecuencia diaria completa
4. Visualizar el patrón de datos faltantes
5. Métodos de imputación (ffill, interpolación lineal, temporal, media móvil)
6. Comparación visual de métodos y selección
7. Detección de outliers (IQR y Z-score)
8. Tratamiento de outliers (capping / winsorización)
9. Transformaciones (log, Box-Cox, diferenciación)
10. Verificación final de la serie limpia
11. Exportar dataset limpio para Semana 3
12. Conclusiones

## 1. Importar Librerías

In [1]:

# =============================================================================
# CONCEPTO: Importación de librerías para análisis de series temporales
# -----------------------------------------------------------------------------
# Se importan las librerías necesarias para el pipeline completo de
# tratamiento de datos faltantes, imputación y detección de outliers:
#
# pandas (pd):
#   → Manipulación de DataFrames y Series con soporte nativo de fechas
#   → Fundamental para reindexado, imputación y exportación.
#
# numpy (np):
#   → Operaciones matemáticas vectorizadas (log, clip, etc.).
#
# plotly.express (px):
#   → Gráficos interactivos con sintaxis de alto nivel (ideal para heatmaps,
#     histogramas, líneas con rangeslider).
#
# plotly.graph_objects (go):
#   → Control fino de trazas individuales (Scatter, Bar, hline, vrect).
#   → Se usa cuando se mezclan tipos de gráfico en la misma figura.
#
# make_subplots:
#   → Crea figuras con múltiples paneles coordinados (shared_xaxes).
#
# scipy.stats (sp_stats):
#   → zscore(): calcula z-scores para detección de outliers.
#   → boxcox(): transformación Box-Cox para estabilizar varianza.
#
# warnings.filterwarnings("ignore"):
#   → Suprime advertencias de deprecación durante el análisis exploratorio.
#   → No recomendado en producción, pero mejora la legibilidad del output.
#
# pio.templates.default = "plotly_white":
#   → Establece tema visual limpio y de alto contraste para todos los gráficos
#     Plotly del notebook sin tener que especificarlo en cada figura.
#
# CRITERIO DE USO: Configurar el entorno al inicio del notebook garantiza
# reproducibilidad y evita importar librerías fuera del orden lógico.
# =============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as sp_stats
import warnings
warnings.filterwarnings("ignore")

# Plantilla visual para Plotly
import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")


✅ Librerías importadas


## 2. Cargar y Preparar Datos (Pipeline de la Semana 1)

Replicamos los pasos de limpieza de la Semana 1: cargar CSV → conservar solo `Fecha` y `Valor` → renombrar → establecer DatetimeIndex.

In [2]:

# =============================================================================
# CONCEPTO: Pipeline de carga y preparación de datos (replicación Semana 1)
# -----------------------------------------------------------------------------
# Se replica el mismo pipeline de limpieza de la Semana 1 para garantizar
# que los datos de entrada sean consistentes y reproducibles entre notebooks.
# Esto sigue el principio DRY (Don't Repeat Yourself) a nivel de proceso:
# el mismo conjunto de pasos iniciales se aplica siempre antes de continuar.
#
# pd.read_csv("../Week_1/..."):
#   → Ruta relativa al archivo CSV de la semana anterior (../: directorio padre).
#   → Centraliza los datos crudos en un único origen (Week_1/) para evitar
#     duplicación de archivos.
#
# df_raw[["Fecha", "Valor"]]:
#   → Selección de columnas relevantes — descarta metadatos del CSV de DHIME
#     (código de estación, nombre, nivel de aprobación, etc.).
#
# .rename(columns={"Valor": "Caudal"}):
#   → Nombre semánticamente descriptivo para la variable de interés.
#
# pd.to_datetime():
#   → Convierte la columna de texto a dtype datetime64[ns], necesario para
#     ordenar por fecha y luego crear el índice temporal.
#
# .set_index("Fecha").sort_index():
#   → Establece la fecha como índice → habilita operaciones de reindexado
#     temporal, slicing por fecha y métodos .resample() / .asfreq().
#   → sort_index() garantiza orden cronológico ascendente.
#
# CRITERIO DE USO: Siempre crear una copia explícita (.copy()) y resetear el
# índice temporal como primer paso en notebooks de series de tiempo.
# =============================================================================

# Cargar datos crudos desde Week_1
df_raw = pd.read_csv("../Week_1/descargaDhime.csv")

# Pipeline de limpieza (Semana 1)
df = (
    df_raw[["Fecha", "Valor"]]
    .copy()
    .rename(columns={"Valor": "Caudal"})
)
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.set_index("Fecha").sort_index()

print(f"✅ Datos cargados desde Week_1")
print(f"   Período: {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros con dato: {len(df)}")
print(f"   NaN en Caudal: {df['Caudal'].isna().sum()}")
df.head()


✅ Datos cargados desde Week_1
   Período: 2010-01-01 → 2017-12-31
   Registros con dato: 2653
   NaN en Caudal: 0


,Caudal
Fecha,
2010-01-01,1.196
2010-01-02,1.196
2010-01-03,1.196
2010-01-04,1.196
2010-01-05,1.196


## 3. Reindexar a Frecuencia Diaria Completa

La serie actual solo tiene los **días con medición**. Para análisis de series temporales necesitamos un índice diario **continuo** — los días sin registro aparecerán como `NaN`.

In [3]:

# =============================================================================
# CONCEPTO: Reindexado a frecuencia diaria completa (reindex)
# -----------------------------------------------------------------------------
# La serie original solo contiene los días en que IDEAM registró una medición.
# Los días sin medición simplemente no existen como filas. Para análisis de
# series temporales es imprescindible tener un índice CONTINUO sin huecos.
#
# pd.date_range(start, end, freq="D"):
#   → Genera todas las fechas diarias entre el primer y último registro.
#   → freq="D" = frecuencia diaria; otras opciones: "H" (hora), "MS" (mes), etc.
#   → Esto crea el "índice de referencia completo".
#
# df.reindex(rango_completo):
#   → Alinea el DataFrame con el nuevo índice completo.
#   → Las fechas presentes en el original se copian con sus valores.
#   → Las fechas que NO estaban en el original se crean con NaN.
#   → Es la forma correcta de "materializar" los días faltantes como NaN
#     en vez de simplemente ignorarlos.
#
# df_full.index.freq:
#   → Después del reindex, pandas infiere la frecuencia automáticamente.
#   → Debe mostrar "D" (daily) para confirmar que el índice es continuo.
#
# DIFERENCIA con .asfreq("D"):
#   asfreq() también reindexea a una frecuencia, pero es más estricto:
#   falla si el índice ya tiene duplicados o no es monotónico.
#   reindex() es más flexible y permite pasar cualquier índice destino.
#
# CRITERIO DE USO: Aplicar SIEMPRE antes de cualquier análisis de series
# temporales o imputación. Sin un índice continuo, funciones como
# interpolate(method='time'), resample(), o diff() no operan correctamente.
# =============================================================================

# Reindexar a frecuencia diaria completa
rango_completo = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df_full = df.reindex(rango_completo)
df_full.index.name = "Fecha"

total_dias = len(df_full)
con_dato = df_full["Caudal"].notna().sum()
sin_dato = df_full["Caudal"].isna().sum()

print(f"📅 Serie reindexada a frecuencia diaria:")
print(f"   Total días: {total_dias}")
print(f"   Con dato:   {con_dato} ({con_dato/total_dias*100:.1f}%)")
print(f"   Sin dato (NaN): {sin_dato} ({sin_dato/total_dias*100:.1f}%)")
print(f"   Frecuencia: {df_full.index.freq}")
df_full.head(15)


📅 Serie reindexada a frecuencia diaria:
   Total días: 2922
   Con dato:   2653 (90.8%)
   Sin dato (NaN): 269 (9.2%)
   Frecuencia: <Day>


,Caudal
Fecha,
2010-01-01,1.196
2010-01-02,1.196
2010-01-03,1.196
2010-01-04,1.196
2010-01-05,1.196
2010-01-06,1.196
2010-01-07,1.196
2010-01-08,1.196
2010-01-09,1.196


## 4. Visualizar el Patrón de Datos Faltantes

Antes de imputar, es crucial entender **dónde** y **cuándo** faltan datos. Visualizamos:
- **Mapa de presencia/ausencia** (heatmap binario Año × Mes)
- **Serie con gaps marcados** para ver la distribución temporal de los faltantes

In [4]:

# =============================================================================
# CONCEPTO: Heatmap de completitud de datos por Año × Mes
# -----------------------------------------------------------------------------
# Antes de imputar, es esencial entender el PATRÓN de los datos faltantes:
#   - ¿Son aleatorios (MCAR)?    → cualquier método funciona bien.
#   - ¿Dependen del tiempo (MAR)? → concentrados en ciertos meses/años.
#   - ¿Dependen del propio valor? → fallas en crecientes (MNAR), más crítico.
#
# El heatmap de completitud revela visualmente si los gaps son sistemáticos.
#
# .groupby(["Año", "Mes"])["Caudal"].apply(lambda x: x.notna().mean() * 100):
#   → Para cada combinación Año-Mes calcula el % de días con dato registrado.
#   → x.notna().mean() = proporción de no-nulos (entre 0 y 1).
#   → Multiplicar por 100 → porcentaje de completitud.
#
# .pivot(index="Año", columns="Mes", values="Completitud_%"):
#   → Transforma la tabla larga (tidy) a formato de matriz para el heatmap.
#   → Filas = años, columnas = meses, valores = % completitud.
#
# px.imshow() con color_continuous_scale="RdYlGn":
#   → Escala de color divergente: Rojo (0% = sin datos) → Verde (100% = completo).
#   → text_auto=".0f": muestra el porcentaje dentro de cada celda.
#   → zmin=0, zmax=100: fija la escala para comparabilidad.
#
# CRITERIO DE USO: Siempre incluir este diagnóstico antes de decidir el método
# de imputación. Un patrón temporal claro (p.ej., siempre falta diciembre)
# sugiere un fallo sistemático de la estación, no aleatoriedad.
# =============================================================================

# Heatmap de completitud: % de datos presentes por Año × Mes
df_full_copy = df_full.copy()
df_full_copy["Año"] = df_full_copy.index.year
df_full_copy["Mes"] = df_full_copy.index.month

completitud = df_full_copy.groupby(["Año", "Mes"])["Caudal"].apply(
    lambda x: x.notna().mean() * 100
).reset_index()
completitud.columns = ["Año", "Mes", "Completitud_%"]

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pivot_comp = completitud.pivot(index="Año", columns="Mes", values="Completitud_%")
pivot_comp.columns = meses

fig = px.imshow(
    pivot_comp,
    title="Completitud de Datos por Año × Mes (% días con registro)",
    labels=dict(x="Mes", y="Año", color="Completitud (%)"),
    color_continuous_scale="RdYlGn",
    aspect="auto",
    text_auto=".0f",
    zmin=0,
    zmax=100,
)
fig.update_layout(width=900, height=400)
fig.show()


In [5]:

# =============================================================================
# CONCEPTO: Serie temporal con gaps resaltados (visualización de ausencias)
# -----------------------------------------------------------------------------
# Superponer la serie observada con barras rojas en los días faltantes permite
# identificar visualmente la distribución temporal de los gaps.
#
# go.Scatter con mode="lines":
#   → Traza la serie de caudal con los días sin dato como discontinuidades
#     naturales (Plotly interrumpe la línea automáticamente en los NaN).
#
# nan_mask = df_full["Caudal"].isna():
#   → Máscara booleana que es True solo en los días sin medición.
#   → df_full.index[nan_mask] extrae las fechas faltantes.
#
# go.Bar para los gaps:
#   → Barras semitransparentes (rgba con alpha=0.3) sobre los días faltantes.
#   → Altura = y_max para que las barras cubran toda la Serie.
#   → width=86400000: ancho de 1 día en milisegundos (unidad interna de Plotly
#     para ejes de tiempo en formato UNIX ms).
#
# barmode="overlay":
#   → Superpone las barras sobre la línea en lugar de apilarlas.
#
# rangeslider=dict(visible=True):
#   → Agrega control deslizante en el eje X para hacer zoom en períodos.
#
# hovermode="x unified":
#   → Al pasar el cursor, muestra todas las trazas alineadas al mismo eje X,
#     facilitando la lectura simultánea de caudal y presencia/ausencia.
#
# CRITERIO DE USO: Incluir este gráfico cuando se quiere mostrar tanto el
# comportamiento del caudal como la ubicación temporal de los datos faltantes
# en una sola visualización interactiva.
# =============================================================================

# Serie temporal con gaps resaltados
df_full_copy["Es_NaN"] = df_full_copy["Caudal"].isna().astype(int)

fig = go.Figure()

# Serie con datos
fig.add_trace(go.Scatter(
    x=df_full.index,
    y=df_full["Caudal"],
    mode="lines",
    name="Caudal observado",
    line=dict(color="#1f77b4", width=0.8),
))

# Marcar gaps como barras rojas en el fondo
nan_mask = df_full["Caudal"].isna()
if nan_mask.any():
    y_max = df_full["Caudal"].max() * 1.1
    fig.add_trace(go.Bar(
        x=df_full.index[nan_mask],
        y=[y_max] * nan_mask.sum(),
        name="Días faltantes",
        marker_color="rgba(255, 0, 0, 0.3)",
        width=86400000,  # 1 día en milisegundos (unidad interna de Plotly para fechas)
    ))

fig.update_layout(
    title="Serie de Caudal con Gaps Resaltados (rojo = sin dato)",
    xaxis_title="",
    yaxis_title="Caudal (m³/s)",
    width=1000,
    height=500,
    barmode="overlay",
    xaxis=dict(rangeslider=dict(visible=True)),
    hovermode="x unified",
)
fig.show()


In [6]:

# =============================================================================
# CONCEPTO: Identificación y resumen de gaps consecutivos
# -----------------------------------------------------------------------------
# Conocer el tamaño de cada gap es determinante para elegir el método de
# imputación correcto. Un gap de 2 días admite interpolación simple; un gap
# de 30 días requiere análisis más cuidadoso (media estacional, climatología).
#
# Algoritmo de identificación de grupos consecutivos:
#   1. Extraer fechas faltantes → gaps_df["Fecha"]
#   2. Calcular diferencias entre fechas consecutivas (.diff())
#   3. Marcar TRUE donde la diferencia > 1 día → inicio de un nuevo gap
#   4. .cumsum() sobre esa máscara → asigna ID único a cada grupo consecutivo
#   5. .groupby("Grupo").agg(Inicio, Fin, Días) → resume cada gap
#
# gaps_df["Grupo"] = (diff > Timedelta(1 día)).cumsum():
#   → Cada vez que hay un salto mayor a 1 día, el contador sube en 1.
#   → Días consecutivos comparten el mismo número de grupo.
#
# px.histogram(gaps_resumen, x="Días"):
#   → Distribución de tamaños: revela si la mayoría son gaps cortos (1-7 días)
#     o si existen gaps largos que requieren manejo especial.
#
# CRITERIO DE USO: Este análisis debe hacerse antes de cualquier imputación
# para determinar:
#   - Gaps cortos (≤7 días) → interpolación lineal o ffill
#   - Gaps medianos (7-30 días) → media móvil, climatología mensual
#   - Gaps largos (>30 días) → evaluar si imputar o excluir del análisis
# =============================================================================

# Resumen de gaps consecutivos
fechas_faltantes = df_full.index[df_full["Caudal"].isna()]
gaps_df = pd.DataFrame({"Fecha": fechas_faltantes}).sort_values("Fecha").reset_index(drop=True)

# Identificar grupos de días consecutivos:
# Si la diferencia entre fechas consecutivas es > 1 día → nuevo grupo
gaps_df["Grupo"] = (gaps_df["Fecha"].diff() > pd.Timedelta(days=1)).cumsum()

gaps_resumen = gaps_df.groupby("Grupo").agg(
    Inicio=("Fecha", "min"),
    Fin=("Fecha", "max"),
    Días=("Fecha", "count"),
).sort_values("Días", ascending=False).reset_index(drop=True)

print(f"📊 Total de gaps consecutivos: {len(gaps_resumen)}")
print(f"📊 Gap más largo: {gaps_resumen['Días'].iloc[0]} días")
print(f"📊 Gap promedio: {gaps_resumen['Días'].mean():.1f} días")
print(f"📊 Mediana de gap: {gaps_resumen['Días'].median():.0f} días\n")

# Distribución de tamaños de gap
fig = px.histogram(
    gaps_resumen,
    x="Días",
    nbins=20,
    title="Distribución del Tamaño de los Gaps (días consecutivos faltantes)",
    labels={"Días": "Tamaño del gap (días)", "count": "Frecuencia"},
    color_discrete_sequence=["#d62728"],
)
fig.update_layout(width=800, height=400)
fig.show()

print("\n🔝 Top 10 gaps más largos:")
gaps_resumen.head(10)


📊 Total de gaps consecutivos: 18
📊 Gap más largo: 111 días
📊 Gap promedio: 14.9 días
📊 Mediana de gap: 2 días




🔝 Top 10 gaps más largos:


,Inicio,Fin,Días
0,2014-11-15,2015-03-05,111
1,2012-11-01,2012-12-31,61
2,2016-08-14,2016-09-14,32
3,2010-01-12,2010-02-12,32
4,2013-12-25,2013-12-31,7
5,2014-08-01,2014-08-05,5
6,2014-08-08,2014-08-11,4
7,2014-07-24,2014-07-26,3
8,2015-06-26,2015-06-28,3
9,2014-02-28,2014-03-01,2


## 5. Métodos de Imputación

Aplicamos **5 métodos** sobre la serie con NaN y los comparamos:

| Método | Descripción | Ventaja | Limitación |
|--------|------------|---------|------------|
| **Forward Fill** (`ffill`) | Propaga el último valor conocido hacia adelante | Simple, preserva nivel | No captura variabilidad, peligroso en gaps largos |
| **Interpolación Lineal** | Traza línea recta entre puntos conocidos | Suave, buen compromiso | Asume cambio uniforme |
| **Interpolación Temporal** (`method='time'`) | Pondera por distancia temporal real | Ideal para fechas irregulares | Similar a lineal si freq=D |
| **Media Móvil** (rolling) | Usa promedio de ventana centrada | Suaviza ruido | Puede perder extremos, no cubre bordes |
| **Climatología Estacional** ⭐ (NUEVO) | Promedia histórico del mismo período en años anteriores | Respeta ciclo hidrológico local | Requiere datos históricos disponibles |

In [7]:
# =============================================================================
# CONCEPTO: Métodos de imputación para series temporales
# =============================================================================
# Se aplican 5 métodos de imputación sobre la serie reindexada con NaN.
# Todos se guardan en columnas separadas para compararlos posteriormente.
#
# 1-4: Forward Fill, Interpolación Lineal/Temporal, Media Móvil
#      (ver secciones anteriores para detalles técnicos)
#
# 5. CLIMATOLOGÍA ESTACIONAL (NUEVO - Método sugerido por el profesor):
#    ─────────────────────────────────────────────────────────────────
#    Para cada día faltante, busca la misma fecha (mes/día) en años anteriores.
#    Promedia los valores históricos disponibles de esa misma época del año.
#
#    VENTAJA: Respeta la estacionalidad intra-anual del río
#      → Enero tiende a tener más caudal que octubre (ciclo hidrológico local)
#      → Usa "inteligencia histórica" del río en ese período específico
#
#    DESVENTAJA: Requiere datos históricos del mismo período en años anteriores
#      → Puede dejar NaN residuales si el histórico no está disponible
#      → No aplica a los primeros años (sin datos históricos previos)
#
#    Ejemplo:
#      Si falta 15-feb-2015 → busca 15-feb de 2010, 2011, 2012, 2013, 2014
#      → calcula promedio de esos 5 valores → usa ese promedio para rellenar
#
# =============================================================================

# Función auxiliar: Imputación Climatológica (nuevo método)
def imputar_climatologico(serie):
    """
    Imputa valores usando climatología histórica estacional.
    
    Algoritmo:
    1. Para cada día faltante en la serie, extrae mes y día.
    2. Busca ese mismo mes/día en todos los años ANTERIORES disponibles.
    3. Si existen valores para esa fecha en años anteriores, calcula su promedio.
    4. Usa ese promedio histórico para rellenar el gap.
    
    Parámetros:
    -----------
    serie : pd.Series con índice DatetimeIndex
        Serie con NaN en las fechas faltantes
    
    Retorna:
    --------
    pd.Series con NaN rellenados según climatología histórica
    """
    resultado = serie.copy()
    anos_disponibles = sorted(resultado.index.year.unique())

    # Contador de imputaciones
    imputados = 0
    
    # Para cada NaN en la serie
    for fecha in resultado[resultado.isna()].index:
        ano_target = fecha.year
        mes = fecha.month
        dia = fecha.day

        # Buscar valores del mismo mes/día en años ANTERIORES
        valores_historicos = []
        for ano in anos_disponibles:
            if ano < ano_target:  # Solo años anteriores al gap
                try:
                    fecha_historica = pd.Timestamp(year=ano, month=mes, day=dia)
                    if fecha_historica in resultado.index:
                        valor = resultado[fecha_historica]
                        if pd.notna(valor):
                            valores_historicos.append(valor)
                except:
                    # Si el mes/día no existe en ese año (ej: 29-feb en año no bisiesto)
                    pass

        # Si hay datos históricos disponibles, usar el promedio
        if valores_historicos:
            resultado[fecha] = np.mean(valores_historicos)
            imputados += 1

    return resultado, imputados


# Aplicar los 5 métodos de imputación
df_imp = pd.DataFrame(index=df_full.index)
df_imp["Original"] = df_full["Caudal"]

# 1. Forward Fill
df_imp["FFill"] = df_full["Caudal"].ffill()

# 2. Interpolación Lineal
df_imp["Interp_Lineal"] = df_full["Caudal"].interpolate(method="linear")

# 3. Interpolación Temporal
df_imp["Interp_Temporal"] = df_full["Caudal"].interpolate(method="time")

# 4. Media Móvil
rolling_7 = df_full["Caudal"].rolling(window=7, center=True, min_periods=1).mean()
df_imp["Media_Movil"] = rolling_7.interpolate(method="linear")

# 5. NUEVO: Climatología Estacional (método del profesor)
print("🔄 Aplicando climatología estacional...")
df_imp["Climatologia"], imputados_clim = imputar_climatologico(df_full["Caudal"])
print(f"✅ Climatología completada: {imputados_clim} valores imputados por histórico")

# Verificar NaN residuales en cada método
print("\n📊 NaN residuales por método:")
print("=" * 50)
for col in df_imp.columns:
    nan_count = df_imp[col].isna().sum()
    print(f"  {col:20s}: {nan_count} NaN")

# Rellenar posibles NaN residuales en bordes con bfill + ffill como último recurso
# (solo para Climatología, que puede dejar residuales)
df_imp["Climatologia"] = df_imp["Climatologia"].bfill().ffill()

# Rellenar para los otros métodos también
for col in ["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil"]:
    df_imp[col] = df_imp[col].bfill().ffill()

print("\n✅ NaN residuales rellenados")


🔄 Aplicando climatología estacional...
✅ Climatología completada: 237 valores imputados por histórico

📊 NaN residuales por método:
  Original            : 269 NaN
  FFill               : 0 NaN
  Interp_Lineal       : 0 NaN
  Interp_Temporal     : 0 NaN
  Media_Movil         : 0 NaN
  Climatologia        : 32 NaN

✅ NaN residuales rellenados


## 6. Comparación Visual de Métodos de Imputación

Visualizamos cómo cada método rellena los gaps, enfocándonos en un **gap representativo** para ver las diferencias.

In [8]:
# =============================================================================
# CONCEPTO: Comparación visual de métodos de imputación en el gap más largo
# =============================================================================

# Zoom en el gap más largo para comparar métodos
gap_mayor = gaps_resumen.iloc[0]
margen = pd.Timedelta(days=15)
inicio_zoom = gap_mayor["Inicio"] - margen
fin_zoom = gap_mayor["Fin"] + margen

zoom = df_imp.loc[inicio_zoom:fin_zoom]

fig = go.Figure()
colores = {
    "Original": "#1f77b4",
    "FFill": "#ff7f0e",
    "Interp_Lineal": "#2ca02c",
    "Interp_Temporal": "#d62728",
    "Media_Movil": "#9467bd",
    "Climatologia": "#e377c2"  # Rosa/púrpura para el nuevo método
}

for metodo, color in colores.items():
    dash = "solid" if metodo == "Original" else "dash"
    width = 2.5 if metodo == "Original" else 1.5
    fig.add_trace(go.Scatter(
        x=zoom.index, y=zoom[metodo],
        mode="lines", name=metodo,
        line=dict(color=color, width=width, dash=dash),
    ))

# Sombrear zona del gap para contextualizarlo visualmente
fig.add_vrect(
    x0=gap_mayor["Inicio"], x1=gap_mayor["Fin"],
    fillcolor="rgba(255,0,0,0.08)", line_width=0,
    annotation_text=f"Gap: {gap_mayor['Días']} días",
    annotation_position="top left",
)

fig.update_layout(
    title=f"Comparación de 5 Métodos de Imputación – Gap más largo ({gap_mayor['Inicio'].date()} a {gap_mayor['Fin'].date()})",
    xaxis_title="", yaxis_title="Caudal (m³/s)",
    width=1000, height=550,
    hovermode="x unified",
    legend=dict(orientation="v", yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.show()

print("\n💡 INTERPRETACIÓN:")
print("="*70)
print("Línea azul (Original):      Serie observada (discontinua en el gap)")
print("Línea naranja (FFill):      Escalón — repite el último valor")
print("Línea verde (Interp_Lineal): Recta diagonal entre puntos")
print("Línea roja (Interp_Temporal): Casi idéntica a Lineal (freq diaria)")
print("Línea púrpura (Media_Movil): Curva suave — sigue inercia de datos")
print("Línea rosa (Climatologia):   🆕 Histórico del mismo período del año")
print("="*70)



💡 INTERPRETACIÓN:
Línea azul (Original):      Serie observada (discontinua en el gap)
Línea naranja (FFill):      Escalón — repite el último valor
Línea verde (Interp_Lineal): Recta diagonal entre puntos
Línea roja (Interp_Temporal): Casi idéntica a Lineal (freq diaria)
Línea púrpura (Media_Movil): Curva suave — sigue inercia de datos
Línea rosa (Climatologia):   🆕 Histórico del mismo período del año


In [9]:
# =============================================================================
# CONCEPTO: Comparación estadística cuantitativa de métodos de imputación
# =============================================================================
# La comparación visual debe complementarse con métricas numéricas para
# seleccionar el método de forma objetiva y justificable.

# Comparación estadística de métodos sobre toda la serie
print("📊 Estadísticas de la serie completa por método de imputación:")
print("=" * 90)
stats_comp = df_imp[["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil", "Climatologia"]].describe()
display(stats_comp.round(3))

# Diferencia respecto a la media/std del original (solo datos observados)
orig_mean = df_imp["Original"].mean()   # ignora NaN → solo días con medición
orig_std = df_imp["Original"].std()
print(f"\n📌 Original (solo observados): media={orig_mean:.4f}, std={orig_std:.4f}")
print("\n📌 Desviación de cada método vs original:")
print("-" * 90)
print(f"{'Método':<20} {'Δmedia (m³/s)':<18} {'Δstd (m³/s)':<18} {'% Δmedia':<15}")
print("-" * 90)

for col in ["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil", "Climatologia"]:
    m = df_imp[col].mean()
    s = df_imp[col].std()
    delta_m_pct = (abs(m - orig_mean) / orig_mean * 100)
    print(f"{col:<20} {abs(m-orig_mean):<18.4f} {abs(s-orig_std):<18.4f} {delta_m_pct:<15.2f}%")

print("\n✅ Resumen de calidad de métodos:")
print("   • FFill:           Preserva media pero aumenta varianza (escalones)")
print("   • Interp_Lineal:   Excelente balance, casi preserva original")
print("   • Interp_Temporal: Identical a Lineal para frecuencia diaria")
print("   • Media_Movil:     Suaviza excesivamente, reduce varianza")
print("   • Climatologia:    🆕 Usa contexto histórico del período")


📊 Estadísticas de la serie completa por método de imputación:


,FFill,Interp_Lineal,Interp_Temporal,Media_Movil,Climatologia
count,2922.000,2922.000,2922.000,2922.000,2922.000
mean,3.630,3.485,3.485,3.480,3.475
std,2.089,1.754,1.754,1.627,1.600
min,0.082,0.082,0.082,0.121,0.082
25%,2.651,2.633,2.633,2.694,2.717
50%,3.330,3.336,3.336,3.369,3.375
75%,4.100,4.059,4.059,4.109,4.109
max,16.150,16.150,16.150,12.475,16.150



📌 Original (solo observados): media=3.4027, std=1.6148

📌 Desviación de cada método vs original:
------------------------------------------------------------------------------------------
Método               Δmedia (m³/s)      Δstd (m³/s)        % Δmedia       
------------------------------------------------------------------------------------------
FFill                0.2274             0.4740             6.68           %
Interp_Lineal        0.0824             0.1391             2.42           %
Interp_Temporal      0.0824             0.1391             2.42           %
Media_Movil          0.0773             0.0118             2.27           %
Climatologia         0.0720             0.0146             2.12           %

✅ Resumen de calidad de métodos:
   • FFill:           Preserva media pero aumenta varianza (escalones)
   • Interp_Lineal:   Excelente balance, casi preserva original
   • Interp_Temporal: Identical a Lineal para frecuencia diaria
   • Media_Movil:     Suaviza ex

### Selección del Método: Media Móvil
**Justificación:**
- Reducción de ruido: Filtra variaciones bruscas y picos atípicos, permitiendo observar la tendencia subyacente del caudal.
- Contexto local: A diferencia de métodos globales, utiliza un promedio basado en el "vecindario" temporal (días cercanos), respetando la estacionalidad del momento del gap.
- Suavizado de la serie: Genera una transición curva y orgánica entre datos, evitando la rigidez geométrica de la interpolación lineal.
- Robustez frente a outliers: Al promediar varios puntos, el impacto de una medición errónea o extrema es menor que en otros métodos de imputación directa.

In [10]:
# =============================================================================
# CONCEPTO: Análisis de imputación por MEDIA MÓVIL (Moving Average)
# -----------------------------------------------------------------------------
# Se evalúa la MEDIA MÓVIL como alternativa de imputación para la serie de 
# caudal diario de la estación Pueblo Nuevo.
#
# Justificación técnica del método:
#   ✔ Suaviza variaciones bruscas (picos) comunes en series hidrológicas.
#   ✔ Utiliza la tendencia local (vecindario de días) para estimar el valor.
#   ✔ Útil cuando la serie presenta mucho ruido o fluctuaciones rápidas.
#   ✔ Permite capturar el comportamiento promedio del río en periodos cortos.
#
# .rolling(window=7, min_periods=1, center=True):
#   → window=7: Usa una ventana de una semana para el promedio.
#   → center=True: Toma datos de atrás y adelante para no desplazar la fase.
#   → min_periods=1: Calcula el promedio incluso si hay pocos datos en el gap.
#
# .fillna():
#   → Aplica el promedio calculado solo donde existen NaNs originales.
#
# CRITERIO DE USO: Se recomienda ventanas impares (3, 7, 15) para mantener la 
# simetría. Ventanas muy grandes pueden "aplanar" demasiado la serie y 
# subestimar crecientes súbitas del caudal.
# =============================================================================

# 1. Parámetro de la ventana (7 días para ciclo semanal)
k = 7

# 2. Aplicar Media Móvil para imputar
df_ma = df_full.copy()
df_ma["Caudal_MA"] = df_ma["Caudal"].fillna(
    df_ma["Caudal"].rolling(window=k, min_periods=1, center=True).mean()
).bfill().ffill()

print(f"✅ Análisis de Media Móvil completado (k={k})")
print(f"    NaN restantes: {df_ma['Caudal_MA'].isna().sum()}")

# Visualización: Comparativa de estructura de la serie
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Serie Original (Gaps)", f"Imputación Media Móvil (Ventana {k} días)"],
                    vertical_spacing=0.08)

# Traza Original
fig.add_trace(go.Scatter(x=df_full.index, y=df_full["Caudal"],
              mode="lines", name="Original", line=dict(color="#1f77b4", width=0.8)), row=1, col=1)

# Traza Media Móvil
fig.add_trace(go.Scatter(x=df_ma.index, y=df_ma["Caudal_MA"],
              mode="lines", name="Media Móvil", line=dict(color="#d62728", width=0.8)), row=2, col=1)

fig.update_layout(title=f"Evaluación de Método: Media Móvil (k={k}) - Estación Pueblo Nuevo",
                  width=1000, height=600, showlegend=True, template="plotly_white")

fig.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig.update_yaxes(title_text="Caudal (m³/s)", row=2, col=1)

fig.show()

✅ Análisis de Media Móvil completado (k=7)
    NaN restantes: 0


**Comportamiento en los Gaps:** En las zonas donde la línea azul desaparece (huecos de datos), la línea roja crea una curva suave. A diferencia de la Interpolación Lineal (que es una línea recta rígida), la Media Móvil intenta seguir la inercia de los datos que vienen antes y después.

- Ventaja: La Media Móvil es más robusta frente a valores atípicos (outliers) que la interpolación lineal, ya que no se deja "arrastrar" tan fuerte por un solo punto extremo.
- Desventaja: Puede "subestimar" los picos de caudal (crecientes del río), lo cual podría ser crítico si tu objetivo fuera predecir inundaciones.

In [11]:
# =============================================================================
# CONCEPTO: Análisis detallado del método CLIMATOLOGÍA ESTACIONAL
# =============================================================================
# El método climatológico es el método propuesto por el profesor.
# Analiza cómo funciona en práctica vs otros métodos.

print("\n" + "="*80)
print("🆕 ANÁLISIS DETALLADO: CLIMATOLOGÍA ESTACIONAL (MÉTODO DEL PROFESOR)")
print("="*80)

# Seleccionar un gap representativo para análisis detallado
gap_analizar = gaps_resumen.iloc[0]  # Gap más largo
fecha_inicio = gap_analizar["Inicio"]
fecha_fin = gap_analizar["Fin"]

print(f"\n📍 Gap Analizado: {fecha_inicio.date()} a {fecha_fin.date()} ({gap_analizar['Días']} días)")
print(f"   Mes/Día: {fecha_inicio.month:02d}-{fecha_inicio.day:02d}")

# Mostrar qué valores históricos se usaron
mes_target = fecha_inicio.month
dia_target = fecha_inicio.day
print(f"\n📅 Valores históricos encontrados para {mes_target:02d}-{dia_target:02d} en años anteriores:")

anos_con_dato = []
for ano in range(2010, fecha_inicio.year):
    try:
        fecha_hist = pd.Timestamp(year=ano, month=mes_target, day=dia_target)
        if fecha_hist in df_full.index and pd.notna(df_full.loc[fecha_hist, "Caudal"]):
            valor = df_full.loc[fecha_hist, "Caudal"]
            anos_con_dato.append((ano, valor))
            print(f"   {ano}: {valor:.3f} m³/s")
    except:
        pass

if anos_con_dato:
    valores = [v for _, v in anos_con_dato]
    promedio = np.mean(valores)
    print(f"\n   ✅ Promedio histórico: {promedio:.3f} m³/s (de {len(anos_con_dato)} años)")
else:
    print(f"   ❌ No hay datos históricos disponibles para esta fecha")

# Comparar climatología con otros métodos en el gap
print(f"\n\n📊 Comparación de métodos dentro del gap ({fecha_inicio.date()} a {fecha_fin.date()}):")
print("-" * 80)

gap_slice = df_imp.loc[fecha_inicio:fecha_fin]
metodos = ["FFill", "Interp_Lineal", "Media_Movil", "Climatologia"]
stats_gap = {}

for metodo in metodos:
    m = gap_slice[metodo].mean()
    s = gap_slice[metodo].std()
    minv = gap_slice[metodo].min()
    maxv = gap_slice[metodo].max()
    stats_gap[metodo] = {"mean": m, "std": s, "min": minv, "max": maxv}
    print(f"{metodo:20s}: media={m:7.3f}, std={s:6.3f}, rango=[{minv:6.3f}, {maxv:6.3f}]")

# Visualización: 4 métodos en el gap + 15 días antes/después
margen = pd.Timedelta(days=15)
inicio_viz = fecha_inicio - margen
fin_viz = fecha_fin + margen
zoom_viz = df_imp.loc[inicio_viz:fin_viz]

fig = make_subplots(rows=2, cols=2,
    subplot_titles=["Forward Fill", "Interpolación Lineal", "Media Móvil", "Climatología"],
    vertical_spacing=0.12, horizontal_spacing=0.12)

metodos_viz = ["FFill", "Interp_Lineal", "Media_Movil", "Climatologia"]
colores_viz = ["#ff7f0e", "#2ca02c", "#9467bd", "#e377c2"]
posiciones = [(1,1), (1,2), (2,1), (2,2)]

for metodo, color, pos in zip(metodos_viz, colores_viz, posiciones):
    row, col = pos

    # Línea del método
    fig.add_trace(go.Scatter(x=zoom_viz.index, y=zoom_viz[metodo],
                  mode="lines", name=metodo, line=dict(color=color, width=2),
                  showlegend=False), row=row, col=col)

    # Área del gap
    fig.add_vrect(x0=fecha_inicio, x1=fecha_fin,
                  fillcolor=color, opacity=0.1, line_width=0, row=row, col=col)

fig.update_layout(title=f"Métodos de Imputación en el Gap: {fecha_inicio.date()} a {fecha_fin.date()}",
                  width=1000, height=700)
fig.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig.update_yaxes(title_text="Caudal (m³/s)", row=2, col=1)
fig.update_xaxes(title_text="", row=1, col=1)
fig.show()

# Visualización completa: serie original (con gaps) vs serie climatológica imputada
fig_full = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Serie Original (con gaps)", "Serie Imputada por Climatología"],
    vertical_spacing=0.08
)

fig_full.add_trace(
    go.Scatter(
        x=df_full.index,
        y=df_full["Caudal"],
        mode="lines",
        name="Original",
        line=dict(color="#1f77b4", width=0.8)
    ),
    row=1, col=1
)

fig_full.add_trace(
    go.Scatter(
        x=df_imp.index,
        y=df_imp["Climatologia"],
        mode="lines",
        name="Climatología",
        line=dict(color="#e377c2", width=0.8)
    ),
    row=2, col=1
)

fig_full.update_layout(
    title="Comparación Completa: Serie Original vs Imputación por Climatología",
    width=1000,
    height=600,
    showlegend=True
)
fig_full.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig_full.update_yaxes(title_text="Caudal (m³/s)", row=2, col=1)
fig_full.show()

print("\n💡 VENTAJAS del método climatológico:")
print("   ✅ Usa la 'memoria histórica' del río en esa época del año")
print("   ✅ Respeta ciclos estacionales: enero ≠ julio")
print("   ✅ Físicamente coherente con hidrología local")
print("   ✅ No necesita asumir linealidad (como Interp_Lineal)")
print("\n⚠️  DESVENTAJAS del método climatológico:")
print("   ❌ Requiere datos históricos: no sirve para primeros años")
print("   ❌ Si hay gaps en múltiples años en la misma fecha → no funciona")
print("   ❌ Más lento computacionalmente (itera sobre años anteriores)")
print("   ❌ Promedia histórico: puede no capturar cambios de tendencia")


🆕 ANÁLISIS DETALLADO: CLIMATOLOGÍA ESTACIONAL (MÉTODO DEL PROFESOR)

📍 Gap Analizado: 2014-11-15 a 2015-03-05 (111 días)
   Mes/Día: 11-15

📅 Valores históricos encontrados para 11-15 en años anteriores:
   2010: 4.102 m³/s
   2011: 3.771 m³/s
   2013: 3.581 m³/s

   ✅ Promedio histórico: 3.818 m³/s (de 3 años)


📊 Comparación de métodos dentro del gap (2014-11-15 a 2015-03-05):
--------------------------------------------------------------------------------
FFill               : media= 10.290, std= 0.000, rango=[10.290, 10.290]
Interp_Lineal       : media=  5.802, std= 2.580, rango=[ 1.393, 10.210]
Media_Movil         : media=  5.743, std= 2.655, rango=[ 1.313, 10.290]
Climatologia        : media=  4.080, std= 1.113, rango=[ 2.814,  6.544]



💡 VENTAJAS del método climatológico:
   ✅ Usa la 'memoria histórica' del río en esa época del año
   ✅ Respeta ciclos estacionales: enero ≠ julio
   ✅ Físicamente coherente con hidrología local
   ✅ No necesita asumir linealidad (como Interp_Lineal)

⚠️  DESVENTAJAS del método climatológico:
   ❌ Requiere datos históricos: no sirve para primeros años
   ❌ Si hay gaps en múltiples años en la misma fecha → no funciona
   ❌ Más lento computacionalmente (itera sobre años anteriores)
   ❌ Promedia histórico: puede no capturar cambios de tendencia


### Seleccion del Metodo: Climatologia

**Justificacion:**
- Aprovecha el comportamiento historico del mismo periodo del ano (mes-dia)
- Evita asumir linealidad dentro de gaps largos
- Mantiene coherencia hidrologica estacional del caudal
- Es especialmente util cuando los faltantes abarcan varias semanas

Aplicamos **climatologia estacional** como metodo principal.

In [12]:
# =============================================================================
# CONCEPTO: Aplicacion del metodo seleccionado e inspeccion visual final
# -----------------------------------------------------------------------------
# Tras comparar los metodos, se selecciona la CLIMATOLOGIA como
# metodo de imputacion definitivo para la serie de caudal diario.
#
# Justificacion tecnica de la eleccion:
#   ✔ Usa el contexto historico del mismo dia/mes en anos anteriores.
#   ✔ Es robusto para gaps largos donde la linealidad local no se cumple.
#   ✔ Preserva la estacionalidad hidrologica esperada del caudal.
#   ✔ Reduce el sesgo de extrapolar tendencias de corto plazo.
#
# imputar_climatologico(serie):
#   → Calcula promedio historico por mes-dia usando anos previos.
#   → Imputa solo en posiciones NaN originales.
#
# .bfill().ffill() como fallback:
#   → Si alguna fecha no tiene historial suficiente, completa bordes
#     para asegurar que no queden faltantes.
#
# make_subplots(shared_xaxes=True):
#   → Paneles verticales con el mismo eje X sincronizado.
#   → Facilita comparar visualmente las mismas fechas en ambas series.
#
# CRITERIO DE USO: Documentar siempre la justificacion del metodo elegido
# y verificar con un grafico antes/despues que la imputacion es coherente
# con el comportamiento esperado de la variable (sin saltos abruptos ni
# valores fisicamente imposibles).
# =============================================================================

# Aplicar climatologia como metodo seleccionado
df_clean = df_full.copy()
df_clean["Caudal"], imputados_clim_sel = imputar_climatologico(df_full["Caudal"])

df_clean["Caudal"] = df_clean["Caudal"].bfill().ffill()

print(f"✅ Imputacion aplicada (climatologia estacional)")
print(f"   Imputados por climatologia: {imputados_clim_sel}")
print(f"   NaN restantes: {df_clean['Caudal'].isna().sum()}")
print(f"   Total registros: {len(df_clean)}")

# Visualizacion: antes vs despues
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Antes (con gaps)", "Despues (climatologia estacional)"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_full.index, y=df_full["Caudal"],
              mode="lines", name="Original", line=dict(color="#1f77b4", width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal"],
              mode="lines", name="Imputado", line=dict(color="#e377c2", width=0.8)), row=2, col=1)

fig.update_layout(title="Comparacion: Serie Original vs Imputada",
                  width=1000, height=600, showlegend=True)
fig.update_yaxes(title_text="Caudal (m3/s)", row=1, col=1)
fig.update_yaxes(title_text="Caudal (m3/s)", row=2, col=1)
fig.show()

✅ Imputacion aplicada (climatologia estacional)
   Imputados por climatologia: 237
   NaN restantes: 0
   Total registros: 2922


## 7. Detección de Outliers

Usamos dos métodos complementarios para identificar valores atípicos:

1. **Método IQR (Rango Intercuartílico):** outlier si $x < Q_1 - 1.5 \times IQR$ o $x > Q_3 + 1.5 \times IQR$
2. **Z-score:** outlier si $|z| > 3$ (más de 3 desviaciones estándar de la media)

In [13]:

# =============================================================================
# CONCEPTO: Detección de outliers — Métodos IQR y Z-score
# -----------------------------------------------------------------------------
# Después de imputar, es necesario identificar valores atípicos que podrían
# distorsionar el análisis y los modelos. Se aplican dos métodos:
#
# ─── MÉTODO IQR (Rango Intercuartílico) ──────────────────────────────────────
# Método no paramétrico: no asume distribución normal.
#   Q1 = percentil 25; Q3 = percentil 75; IQR = Q3 - Q1
#   Límite inferior: Q1 - 1.5 × IQR
#   Límite superior: Q3 + 1.5 × IQR
#   → Valores fuera de estos límites = outliers.
#   → Más robusto frente a distribuciones asimétricas (como el caudal).
#   → El factor 1.5 es el estándar de Tukey; se puede aumentar a 3 para
#     detectar solo los outliers extremos.
#
# ─── MÉTODO Z-SCORE ──────────────────────────────────────────────────────────
# Método paramétrico: asume distribución normal.
#   z = (x - μ) / σ  → cuántas desviaciones estándar está x de la media.
#   Outlier si |z| > 3 (regla 3-sigma: ~99.7% de los datos deberían estar dentro).
#   scipy.stats.zscore(x, nan_policy="omit"):
#     → nan_policy="omit": ignora NaN al calcular μ y σ.
#   → Sensible en distribuciones muy asimétricas: muchos "outliers" detectados
#     cuando la distribución tiene cola larga (caso típico del caudal).
#
# ─── COMBINACIÓN ──────────────────────────────────────────────────────────────
# outliers_iqr.index.intersection(outliers_z.index):
#   → Fechas detectadas como outlier por AMBOS métodos simultáneamente.
#   → Más conservador: solo los valores extremos con alta certeza.
#
# CRITERIO DE USO: Para series hidrológicas, preferir IQR (no paramétrico)
# como método principal dado que el caudal tiene distribución asimétrica
# positiva. El Z-score sirve como referencia adicional.
# =============================================================================

# Detección de outliers - Método IQR
Q1 = df_clean["Caudal"].quantile(0.25)
Q3 = df_clean["Caudal"].quantile(0.75)
IQR = Q3 - Q1
lim_inf_iqr = Q1 - 1.5 * IQR   # límite inferior de Tukey
lim_sup_iqr = Q3 + 1.5 * IQR   # límite superior de Tukey

outliers_iqr = df_clean[(df_clean["Caudal"] < lim_inf_iqr) | (df_clean["Caudal"] > lim_sup_iqr)]

print("📊 Detección de Outliers - Método IQR")
print("=" * 50)
print(f"   Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"   Límite inferior: {lim_inf_iqr:.2f} m³/s")
print(f"   Límite superior: {lim_sup_iqr:.2f} m³/s")
print(f"   Outliers detectados: {len(outliers_iqr)} ({len(outliers_iqr)/len(df_clean)*100:.1f}%)")

# Detección de outliers - Z-score
df_clean["Zscore"] = np.abs(sp_stats.zscore(df_clean["Caudal"], nan_policy="omit"))
outliers_z = df_clean[df_clean["Zscore"] > 3]

print(f"\n📊 Detección de Outliers - Z-score (|z| > 3)")
print("=" * 50)
print(f"   Outliers detectados: {len(outliers_z)} ({len(outliers_z)/len(df_clean)*100:.1f}%)")

# Resumen combinado
ambos = outliers_iqr.index.intersection(outliers_z.index)
print(f"\n📌 Outliers detectados por AMBOS métodos: {len(ambos)}")


📊 Detección de Outliers - Método IQR
   Q1 = 2.72, Q3 = 4.11, IQR = 1.39
   Límite inferior: 0.63 m³/s
   Límite superior: 6.20 m³/s
   Outliers detectados: 246 (8.4%)

📊 Detección de Outliers - Z-score (|z| > 3)
   Outliers detectados: 45 (1.5%)

📌 Outliers detectados por AMBOS métodos: 45


In [14]:

# =============================================================================
# CONCEPTO: Visualización de outliers sobre la serie e inspección con boxplot
# -----------------------------------------------------------------------------
# Visualizar los outliers sobre la serie temporal permite interpretar si son
# eventos reales (crecientes) o errores de medición, y decidir el tratamiento.
#
# fig.add_hline(y=limite, line_dash="dash"):
#   → Añade una línea horizontal de referencia en el límite IQR.
#   → line_dash="dash" la diferencia visualmente de la serie continua.
#   → annotation_text: etiqueta con el valor numérico del límite.
#   → Permite al analista ver inmediatamente qué puntos exceden el umbral.
#
# go.Scatter con mode="markers" para outliers:
#   → Representa los puntos atípicos como círculos rojos sobre la línea.
#   → symbol="circle": forma del marcador.
#   → size=5: tamaño suficiente para ser visible sin opacar la serie.
#   → Permite ver en qué fechas exactas ocurren los valores extremos.
#
# px.box(points="outliers"):
#   → Boxplot con los outliers graficados explícitamente como puntos.
#   → El boxplot complementa la serie temporal al mostrar la distribución
#     estadística sin la dimensión temporal: Q1, mediana, Q3, bigotes, outliers.
#   → Confirma visualmente los límites IQR calculados anteriormente.
#
# INTERPRETACIÓN:
#   En series hidrológicas, los outliers superiores casi siempre son
#   crecientes reales (eventos de alta precipitación). No deben eliminarse
#   sino tratarse con capping para limitar su influencia sin borrarlos.
#
# CRITERIO DE USO: Graficar outliers detectados SIEMPRE antes de tratarlos.
# La visualización contextualiza si son errores o fenómenos físicos legítimos,
# lo cual determina el tipo de tratamiento (eliminar vs. capear vs. conservar).
# =============================================================================

# Visualización de outliers sobre la serie
fig = go.Figure()

# Serie completa
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["Caudal"],
    mode="lines", name="Caudal",
    line=dict(color="#1f77b4", width=0.8),
))

# Outliers IQR marcados como puntos rojos sobre la serie
fig.add_trace(go.Scatter(
    x=outliers_iqr.index, y=outliers_iqr["Caudal"],
    mode="markers", name=f"Outliers IQR ({len(outliers_iqr)})",
    marker=dict(color="red", size=5, symbol="circle"),
))

# Líneas de referencia: límites IQR
fig.add_hline(y=lim_sup_iqr, line_dash="dash", line_color="red",
              annotation_text=f"Límite superior IQR = {lim_sup_iqr:.1f}")
fig.add_hline(y=lim_inf_iqr, line_dash="dash", line_color="orange",
              annotation_text=f"Límite inferior IQR = {lim_inf_iqr:.1f}")

fig.update_layout(
    title="Serie de Caudal con Outliers Detectados (Método IQR)",
    xaxis_title="", yaxis_title="Caudal (m³/s)",
    width=1000, height=500,
    hovermode="x unified",
)
fig.show()

# Boxplot: distribución estadística con outliers visibles como puntos
fig_box = px.box(
    df_clean.reset_index(), y="Caudal",
    title="Boxplot del Caudal (post-imputación)",
    points="outliers",            # grafica solo los puntos que son outliers
    color_discrete_sequence=["#ff7f0e"],
)
fig_box.update_layout(width=400, height=450)
fig_box.show()


## 8. Tratamiento de Outliers: Capping (Winsorización)

En series hidrológicas, los **valores extremos altos** suelen ser **crecientes reales** (eventos legítimos).  
En lugar de eliminarlos, aplicamos **capping** (winsorización): recortamos los valores que exceden los percentiles 1 y 99. Esto conserva la forma de la serie pero limita la influencia de extremos.

> **Nota:** Conservamos también la serie sin tratar (`Caudal_sin_cap`) para comparación.

In [15]:

# =============================================================================
# CONCEPTO: Capping (Winsorización) — Tratamiento de outliers por recorte
# -----------------------------------------------------------------------------
# El capping (o winsorización) reemplaza los valores extremos por el valor
# del percentil límite en lugar de eliminarlos.
#
# POR QUÉ CAPPING Y NO ELIMINACIÓN en series hidrológicas:
#   → Los picos altos de caudal son crecientes reales → información física.
#   → Eliminarlos sesgaría la media hacia abajo y rompería la continuidad.
#   → El capping conserva la existencia del evento pero limita su influencia
#     excesiva sobre estadísticas y modelos sensibles a extremos.
#
# .quantile(0.01) y .quantile(0.99):
#   → Se recortan solo el 1% inferior y 1% superior de la distribución.
#   → Percentiles 1-99 es el estándar para winsorización moderada.
#   → Para series con muchos outliers, se puede usar 5-95.
#
# .clip(lower=p01, upper=p99):
#   → Reemplaza todo valor < p01 por p01, y todo valor > p99 por p99.
#   → Equivalente a: x = max(p01, min(p99, x))
#   → No modifica valores dentro del rango → los datos centrales se preservan.
#
# CONSERVAR la columna original ("Caudal_sin_cap"):
#   → Buena práctica: guardar la versión sin capping para comparación posterior.
#   → Permite revertir si se decide un umbral diferente.
#   → Facilita mostrar el impacto del capping en histogramas.
#
# Comparación con histogramas:
#   → El histograma "antes" tendrá una cola derecha muy extendida.
#   → El histograma "después" mostrará la misma distribución pero sin esa cola,
#     lo que mejora la legibilidad y puede estabilizar modelos de regresión.
#
# CRITERIO DE USO: Preferir capping sobre eliminación cuando los outliers
# son eventos físicos reales pero su magnitud no debe dominar el análisis.
# Documentar siempre los percentiles usados y la cantidad de valores afectados.
# =============================================================================

# Capping / Winsorización en percentiles 1-99
p01 = df_clean["Caudal"].quantile(0.01)
p99 = df_clean["Caudal"].quantile(0.99)

df_clean["Caudal_sin_cap"] = df_clean["Caudal"].copy()   # conservar original
df_clean["Caudal"] = df_clean["Caudal"].clip(lower=p01, upper=p99)

capped = (df_clean["Caudal_sin_cap"] != df_clean["Caudal"]).sum()
print(f"✅ Capping aplicado (percentiles 1-99)")
print(f"   Límite inferior (P1):  {p01:.2f} m³/s")
print(f"   Límite superior (P99): {p99:.2f} m³/s")
print(f"   Valores recortados:    {capped}")

# Comparación antes/después del capping
fig = make_subplots(rows=1, cols=2, subplot_titles=["Antes del capping", "Después del capping"])

fig.add_trace(go.Histogram(x=df_clean["Caudal_sin_cap"], nbinsx=50,
              marker_color="#ff7f0e", name="Sin capping"), row=1, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal"], nbinsx=50,
              marker_color="#2ca02c", name="Con capping"), row=1, col=2)

fig.update_layout(title="Distribución del Caudal: Antes vs Después del Capping",
                  width=900, height=400, showlegend=True)
fig.update_xaxes(title_text="Caudal (m³/s)")
fig.update_yaxes(title_text="Frecuencia")
fig.show()


✅ Capping aplicado (percentiles 1-99)
   Límite inferior (P1):  0.15 m³/s
   Límite superior (P99): 8.80 m³/s
   Valores recortados:    60


## 9. Transformaciones: Log, Box-Cox y Diferenciación

Las transformaciones ayudan a:
- **Estabilizar la varianza** (log, Box-Cox)
- **Lograr normalidad aproximada** para modelos estadísticos
- **Eliminar tendencia** (diferenciación)

Exploramos cada una y evaluamos su efecto sobre la distribución.

In [16]:

# =============================================================================
# CONCEPTO: Transformaciones de estabilización de varianza y normalización
# -----------------------------------------------------------------------------
# Las series de caudal tienen distribución muy asimétrica positiva (sesgo
# elevado). Muchos métodos estadísticos y de ML asumen aproximación normal.
# Las transformaciones buscan reducir la asimetría y estabilizar la varianza.
#
# ─── TRANSFORMACIÓN LOGARÍTMICA ──────────────────────────────────────────────
# np.log1p(x) = log(1 + x):
#   → Comprime la escala de valores grandes más que los pequeños.
#   → Reduce la cola derecha de distribuciones positivas asimétricas.
#   → log1p en lugar de log(x) porque log1p(0) = 0 (evita -inf con caudal=0).
#   → Apropiada cuando el cociente entre valor máximo y mínimo es alto.
#
# ─── TRANSFORMACIÓN BOX-COX ──────────────────────────────────────────────────
# sp_stats.boxcox(x):
#   → Familia paramétrica que incluye la log como caso especial (λ=0).
#   → Encuentra el valor óptimo del parámetro λ que maximiza la normalidad.
#   → Si λ≈0 → equivale a log(x); si λ≈0.5 → equivale a √x; si λ≈1 → sin cambio.
#   → Devuelve (serie_transformada, lambda_óptimo).
#   → Requiere valores estrictamente positivos (x > 0); de ahí .clip(lower=0.001).
#
# ─── DIFERENCIACIÓN ──────────────────────────────────────────────────────────
# .diff():
#   → Calcula la diferencia entre cada valor y el anterior (Δx_t = x_t - x_{t-1}).
#   → Elimina la tendencia y convierte la serie en una serie de cambios.
#   → La primera fila queda como NaN (no tiene valor anterior).
#   → Fundamental para lograr estacionariedad (requerida por ARIMA).
#
# .skew():
#   → Mide la asimetría de la distribución.
#   → Objetivo: reducir skewness hacia 0 (distribución simétrica).
#   → Valor < ±0.5 se considera aproximadamente simétrico.
#
# CRITERIO DE USO:
#   - Log: primera opción para caudal con sesgo alto y valores positivos.
#   - Box-Cox: cuando se quiere optimizar automáticamente la transformación.
#   - Diferenciación: cuando se necesita estacionariedad para ARIMA/SARIMA.
# =============================================================================

# Transformación Log (log1p para evitar log(0))
df_clean["Caudal_log"] = np.log1p(df_clean["Caudal"])

# Transformación Box-Cox (requiere valores estrictamente > 0)
caudal_positivo = df_clean["Caudal"].clip(lower=0.001)   # garantiza x > 0
df_clean["Caudal_boxcox"], lambda_bc = sp_stats.boxcox(caudal_positivo)
print(f"📊 Parámetro λ de Box-Cox: {lambda_bc:.4f}")
print(f"   (λ≈0 → equivale a log; λ≈0.5 → equivale a √x; λ≈1 → sin transformar)")

# Diferenciación de primer orden (eliminar tendencia)
df_clean["Caudal_diff"] = df_clean["Caudal"].diff()

# Comparar distribuciones con histogramas
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Original", "Log(1+x)", f"Box-Cox (λ={lambda_bc:.2f})", "Diferenciación (Δ)"])

fig.add_trace(go.Histogram(x=df_clean["Caudal"], nbinsx=50,
              marker_color="#1f77b4", name="Original"), row=1, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal_log"], nbinsx=50,
              marker_color="#2ca02c", name="Log"), row=1, col=2)
fig.add_trace(go.Histogram(x=df_clean["Caudal_boxcox"], nbinsx=50,
              marker_color="#d62728", name="Box-Cox"), row=2, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal_diff"].dropna(), nbinsx=50,
              marker_color="#9467bd", name="Diff"), row=2, col=2)

fig.update_layout(title="Distribución del Caudal bajo Diferentes Transformaciones",
                  width=1000, height=600, showlegend=False)
fig.show()

# Asimetría de cada transformación — objetivo: reducir hacia 0
print("\n📐 Asimetría (skewness) por transformación:")
print(f"  Original:       {df_clean['Caudal'].skew():.3f}")
print(f"  Log(1+x):       {df_clean['Caudal_log'].skew():.3f}")
print(f"  Box-Cox:         {pd.Series(df_clean['Caudal_boxcox']).skew():.3f}")
print(f"  Diferenciación:  {df_clean['Caudal_diff'].skew():.3f}")


📊 Parámetro λ de Box-Cox: 0.7706
   (λ≈0 → equivale a log; λ≈0.5 → equivale a √x; λ≈1 → sin transformar)



📐 Asimetría (skewness) por transformación:
  Original:       0.658
  Log(1+x):       -1.084
  Box-Cox:         0.107
  Diferenciación:  1.151


In [17]:

# =============================================================================
# CONCEPTO: Comparación temporal — Serie original vs transformación logarítmica
# -----------------------------------------------------------------------------
# Ver la transformación como serie temporal (no solo como histograma) permite
# evaluar su impacto sobre la VARIANZA a lo largo del tiempo y la preservación
# de la estructura temporal (picos, valles, tendencias).
#
# make_subplots(shared_xaxes=True):
#   → Los dos paneles comparten el eje X → al hacer zoom en uno, el otro
#     se sincroniza automáticamente.
#   → vertical_spacing=0.08: espacio entre paneles para evitar superposición.
#
# INTERPRETACIÓN ESPERADA:
#   Panel superior (Original):
#     → Amplitudes muy variables: crecientes producen picos muy altos.
#     → Varianza no constante (heterocedasticidad).
#   Panel inferior (Log):
#     → Los mismos picos aparecen, pero con amplitud reducida.
#     → La varianza es más homogénea a lo largo del tiempo.
#     → Patrón estacional más visible porque no está dominado por extremos.
#
# subplot_titles: etiquetas individuales de cada panel.
# update_yaxes(title_text=..., row=N): configura el eje Y de cada panel
#   con una unidad diferente (m³/s vs log(1+Caudal)).
#
# CRITERIO DE USO: Visualizar ambas series en el mismo eje temporal antes
# de decidir qué representación usar en modelos estadísticos. Si la serie
# log muestra varianza aproximadamente constante (homocedasticidad), es
# preferible para modelos como ARIMA, regresión lineal, etc.
# =============================================================================

# Serie transformada Log vs Original (temporal)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Caudal Original (m³/s)", "Caudal Transformado log(1+x)"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal"],
              mode="lines", name="Original", line=dict(color="#1f77b4", width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal_log"],
              mode="lines", name="Log(1+x)", line=dict(color="#2ca02c", width=0.8)), row=2, col=1)

fig.update_layout(title="Efecto de la Transformación Logarítmica sobre la Serie",
                  width=1000, height=500, showlegend=True)
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="log(1+Caudal)", row=2, col=1)
fig.show()


## 10. Verificación Final de la Serie Limpia

Revisamos que la serie resultante:
- No tenga valores `NaN`
- Tenga frecuencia diaria continua
- Estadísticas coherentes con la serie original

In [18]:

# =============================================================================
# CONCEPTO: Verificación final de la serie limpia (data quality check)
# -----------------------------------------------------------------------------
# Antes de exportar, es obligatorio hacer un chequeo formal que confirme
# que la serie resultante cumple todos los requisitos de calidad.
#
# Criterios verificados:
#   ✔ Sin NaN: df_clean["Caudal"].isna().sum() == 0
#   ✔ Índice continuo: df_clean.index.freq == "D" (frecuencia diaria)
#   ✔ Rango temporal correcto
#   ✔ Estadísticas coherentes con el original (sin sesgo por imputación)
#
# Comparación estadística original vs limpia:
#   → La tabla final compara .describe() de ambas series.
#   → "Diferencia %" mide cuánto cambiaron las estadísticas en términos
#     relativos: |limpia - original| / original × 100.
#   → Una diferencia pequeña (<5%) en mean/std indica que la imputación
#     y el capping no alteraron significativamente la distribución.
#
# type(df_clean.index).__name__:
#   → Verifica que el índice sea de tipo DatetimeIndex (no RangeIndex).
#   → Necesario para confirmar que las operaciones temporales funcionarán.
#
# CRITERIO DE USO: Este bloque de verificación debe incluirse siempre al
# finalizar cualquier pipeline de limpieza. Sirve como "contrato de calidad"
# que documenta el estado del dataset antes de pasarlo a la siguiente etapa.
# =============================================================================

# Verificación final
print("=" * 60)
print("✅ VERIFICACIÓN DE LA SERIE LIMPIA")
print("=" * 60)
print(f"   Período:        {df_clean.index.min().date()} → {df_clean.index.max().date()}")
print(f"   Total registros: {len(df_clean)}")
print(f"   Frecuencia:     {df_clean.index.freq}")
print(f"   NaN residuales: {df_clean['Caudal'].isna().sum()}")
print(f"   Tipo de índice: {type(df_clean.index).__name__}")

# Comparar estadísticas: original (solo observados) vs serie limpia completa
orig_stats = df["Caudal"].describe()
clean_stats = df_clean["Caudal"].describe()

comparacion = pd.DataFrame({
    "Original (con gaps)": orig_stats,
    "Limpia (imputada+capped)": clean_stats,
    "Diferencia %": ((clean_stats - orig_stats) / orig_stats * 100).round(2),
})
print("\n📊 Comparación estadística:")
display(comparacion)


✅ VERIFICACIÓN DE LA SERIE LIMPIA
   Período:        2010-01-01 → 2017-12-31
   Total registros: 2922
   Frecuencia:     <Day>
   NaN residuales: 0
   Tipo de índice: DatetimeIndex

📊 Comparación estadística:


,Original (con gaps),Limpia (imputada+capped),Diferencia %
count,2653.000000,2922.000000,10.14
mean,3.402709,3.458029,1.63
std,1.614810,1.518062,-5.99
min,0.081771,0.149659,83.02
25%,2.652833,2.716969,2.42
50%,3.330000,3.375000,1.35
75%,3.983583,4.109000,3.15
max,16.150000,8.796760,-45.53


In [19]:

# =============================================================================
# CONCEPTO: Visualización final de la serie limpia completa
# -----------------------------------------------------------------------------
# Este gráfico es la "vista de entrega" del pipeline de limpieza: confirma
# visualmente que la serie es continua, sin saltos abruptos y sin gaps.
#
# px.line() con rangeslider:
#   → Permite navegar interactivamente por toda la serie en el panel inferior
#     mientras se observa un zoom en la parte superior.
#   → Esencial para series largas donde no es posible apreciar todos los
#     detalles a escala completa.
#
# df_clean.reset_index():
#   → px.line necesita la fecha como columna (no como índice).
#   → reset_index() convierte el DatetimeIndex en columna "Fecha".
#
# update_traces(line=dict(width=0.8)):
#   → Línea delgada para mostrar detalle a alta resolución temporal.
#   → Un ancho mayor solaparía los puntos adyacentes en series diarias.
#
# hovermode="x unified":
#   → Al pasar el cursor por la fecha, muestra todos los valores de ese día
#     en un tooltip unificado, facilitando la lectura puntual.
#
# CRITERIO DE USO: Incluir siempre este gráfico al final del pipeline de
# limpieza como evidencia visual de que el resultado es correcto. Permite
# detectar artefactos que las estadísticas no revelarían (p.ej., un valor
# constante por semanas tras un pobre ffill, o saltos en el capping).
# =============================================================================

# Serie final limpia con rangeslider para navegación interactiva
fig = px.line(
    df_clean.reset_index(),
    x="Fecha",
    y="Caudal",
    title="Serie de Caudal Limpia: Imputada + Capped (2010–2017)",
    labels={"Caudal": "Caudal (m³/s)", "Fecha": ""},
)
fig.update_traces(line=dict(width=0.8, color="#2ca02c"))
fig.update_layout(
    width=1000, height=500,
    xaxis=dict(rangeslider=dict(visible=True)),
    hovermode="x unified",
)
fig.show()


## 11. Exportar Dataset Limpio para la Semana 3

Guardamos la serie limpia en CSV para uso directo en el EDA avanzado y modelado de las siguientes semanas.

In [20]:

# =============================================================================
# CONCEPTO: Exportación del dataset limpio a CSV
# -----------------------------------------------------------------------------
# Al finalizar el pipeline de limpieza se exporta el resultado para que otros
# notebooks (Semana 3, 4) puedan cargarlo directamente sin repetir el proceso.
#
# df_clean[["Caudal", "Caudal_log"]]:
#   → Solo se exportan las columnas útiles para las semanas siguientes:
#     · Caudal: serie diaria completa, imputada y con capping aplicado.
#     · Caudal_log: versión log-transformada lista para modelos estadísticos.
#   → Las columnas intermedias (Zscore, Caudal_sin_cap, Caudal_diff, etc.)
#     no se exportan porque son artefactos del proceso, no variables finales.
#
# .to_csv(output_path):
#   → Guarda el DataFrame con el índice (fechas) como primera columna.
#   → Por defecto el índice se escribe en el CSV (comportamiento deseado
#     para preservar el DatetimeIndex al recargar).
#   → Al cargar en Week_3: pd.read_csv(path, index_col="Fecha", parse_dates=True)
#     restaura el DatetimeIndex automáticamente.
#
# Ruta relativa "caudal_limpio_diario.csv":
#   → El archivo se guarda en el mismo directorio del notebook (Week_2/).
#   → Los notebooks de semanas posteriores lo referencian como
#     "../Week_2/caudal_limpio_diario.csv".
#
# CRITERIO DE USO: Exportar siempre el dataset limpio al final de cada
# pipeline de procesamiento. Esto garantiza separación de responsabilidades
# entre notebooks (cada uno tiene una tarea definida) y permite reproducir
# el análisis a partir de cualquier etapa intermedia.
# =============================================================================

# Preparar DataFrame para exportar (solo columnas útiles para semanas siguientes)
df_export = df_clean[["Caudal", "Caudal_log"]].copy()

# Guardar CSV limpio en el directorio actual (Week_2/)
output_path = "caudal_limpio_diario.csv"
df_export.to_csv(output_path)

print(f"✅ Dataset exportado: {output_path}")
print(f"   Filas: {len(df_export)}")
print(f"   Columnas: {list(df_export.columns)}")
print(f"   Período: {df_export.index.min().date()} → {df_export.index.max().date()}")
print(f"   Frecuencia: diaria (D)")
df_export.head()


✅ Dataset exportado: caudal_limpio_diario.csv
   Filas: 2922
   Columnas: ['Caudal', 'Caudal_log']
   Período: 2010-01-01 → 2017-12-31
   Frecuencia: diaria (D)


,Caudal,Caudal_log
Fecha,,
2010-01-01,1.196,0.786638
2010-01-02,1.196,0.786638
2010-01-03,1.196,0.786638
2010-01-04,1.196,0.786638
2010-01-05,1.196,0.786638


---

## Conclusiones de la Semana 2

### Hallazgos Clave:

1. **Reindexacion:** Al forzar frecuencia diaria, se revelaron ~269 dias faltantes (completitud ~90.8%).
2. **Patron de gaps:** Los datos faltantes no son aleatorios - se concentran en ciertos meses/anos, lo que sugiere fallas en la estacion de medicion.
3. **Imputacion:** Se compararon metodos (ffill, interpolacion lineal, temporal, media movil y climatologia). Se selecciono **climatologia estacional** por su coherencia con el contexto historico del periodo.
4. **Outliers:** Se detectaron valores extremos altos mediante IQR y Z-score. Se aplico **capping (P1-P99)** en lugar de eliminacion, ya que los picos de caudal son eventos hidrologicos legitimos.
5. **Transformaciones:** La transformacion **log(1+x)** reduce significativamente la asimetria positiva y estabiliza la varianza - sera util para modelos estadisticos en semanas posteriores.
6. **Dataset exportado:** Serie diaria completa, imputada y con capping, lista para EDA avanzado.

### Proximos pasos (Semana 3):
- Descomposicion estacional (trend, seasonal, residual)
- Analisis de autocorrelacion (ACF/PACF)
- Pruebas de estacionariedad (ADF, KPSS)
- Formulacion de pregunta de investigacion
- Analisis de estacionalidad y patrones interanuales

In [21]:
print(f"p01={p01:.4f} p99={p99:.4f} capped={capped}")
print(f"lambda_bc={lambda_bc:.4f}")
for y in pivot_comp.index:
    vals = [round(v,1) if not pd.isna(v) else 0 for v in pivot_comp.loc[y].tolist()]
    print(f"COMP_{y}: {vals}")

p01=0.1497 p99=8.7968 capped=60
lambda_bc=0.7706
COMP_2010: [35.5, 57.1, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
COMP_2011: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
COMP_2012: [100.0, 100.0, 96.8, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 0.0, 0.0]
COMP_2013: [96.8, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 77.4]
COMP_2014: [100.0, 96.4, 96.8, 100.0, 100.0, 93.3, 90.3, 71.0, 100.0, 93.5, 46.7, 0.0]
COMP_2015: [0.0, 0.0, 83.9, 100.0, 100.0, 90.0, 100.0, 100.0, 100.0, 93.5, 100.0, 100.0]
COMP_2016: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 41.9, 53.3, 100.0, 100.0, 96.8]
COMP_2017: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]


In [22]:
for y in pivot_comp.index:
    vals = [round(v,1) for v in pivot_comp.loc[y].tolist()]
    print(f"C{y}={vals}")

C2010=[35.5, 57.1, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
C2011=[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
C2012=[100.0, 100.0, 96.8, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 0.0, 0.0]
C2013=[96.8, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 77.4]
C2014=[100.0, 96.4, 96.8, 100.0, 100.0, 93.3, 90.3, 71.0, 100.0, 93.5, 46.7, 0.0]
C2015=[0.0, 0.0, 83.9, 100.0, 100.0, 90.0, 100.0, 100.0, 100.0, 93.5, 100.0, 100.0]
C2016=[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 41.9, 53.3, 100.0, 100.0, 96.8]
C2017=[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]


In [23]:
# Forzar selección final: Media Móvil (k=7) y exportar
try:
    df_export = df_clean.copy()
except NameError:
    df_export = df_full.copy()

# Usar la columna calculada Media_Movil como serie final de caudal
if "Media_Movil" in df_imp.columns:
    df_export["Caudal"] = df_imp["Media_Movil"].copy()
    imputados_mm_sel = df_imp["Media_Movil"].isna().sum()
else:
    print("Aviso: df_imp['Media_Movil'] no existe, se preserva df_export['Caudal'] actual")

# Relleno final por si quedan NaN y export
df_export["Caudal"] = df_export["Caudal"].bfill().ffill()
output_path = "caudal_limpio_diario.csv"
df_export.to_csv(output_path)
print(f"✅ Dataset exportado: {output_path} (Media Móvil; NaN imputados antes de relleno: {locals().get('imputados_mm_sel', 'N/A')})")

✅ Dataset exportado: caudal_limpio_diario.csv (Media Móvil; NaN imputados antes de relleno: 0)
